# Act 1 — EDA and data storytelling (PVDAQ)

Goals:
- Load PVDAQ CSV(s) from `data/raw/`
- Summarize missingness, cadence, and time coverage
- Visualize seasonal and daily patterns and correlations (after NSRDB join in Act 2)

Run `python scripts/compare_pvdaq_facilities.py` after adding CSVs to pick `config/selected_facility.yaml`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data.pvdaq import compute_facility_stats, load_all_raw_csvs
from src.viz.plots import plot_daily_median_profile, set_plot_style

raw_dir = ROOT / "data" / "raw"
frames = load_all_raw_csvs(raw_dir)
if not frames:
    raise FileNotFoundError(f"Add PVDAQ CSV files to {raw_dir} (see data/raw/README.md)")

list(frames.keys())

In [ ]:
cfg_path = ROOT / "config" / "selected_facility.yaml"
if cfg_path.exists():
    text = cfg_path.read_text(encoding="utf-8")
    for line in text.splitlines():
        if line.strip().startswith("system_id:"):
            chosen = line.split(":", 1)[1].strip()
            break
    else:
        chosen = None
else:
    chosen = None

if chosen in (None, "null", ""):
    chosen_sid = next(iter(frames.keys()))
else:
    chosen_sid = int(chosen) if str(chosen).isdigit() else chosen

from dataclasses import asdict

df = frames[chosen_sid]
stats = compute_facility_stats(df)
asdict(stats)

In [ ]:
set_plot_style()
ax = plot_daily_median_profile(df)
ax.figure.savefig(ROOT / "reports" / "figures" / "median_daily_profile.png", dpi=150, bbox_inches="tight")